In [1]:
%pip install -q google-genai

StatementMeta(, 9dba556d-42dd-40cc-b44d-db297855b545, 7, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /nfs4/pyenv-f2bf1101-c8c1-45b4-bca1-061f5ca27c11/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
import time, json
from PIL import Image
from google import genai
from pyspark.sql import functions as F

api_key = notebookutils.fs.head("Files/config/gemini_api_key.txt").strip()
client = genai.Client(api_key=api_key)

# Mismo modelo y mismo prompt que validé en 02_validacion_gemini (error medio 6,6 pp)
MODELO = "gemini-3.5-flash-lite"
VERSION_PROMPT = "v2"
PROMPT = """Esta es una imagen aérea (ortofoto) vista desde arriba, de unos 300 x 300 metros.
1) Clasifica la parcela en una de estas fases de obra:
   terreno_sin_tocar, movimiento_de_tierras, estructuras_sin_paneles, paneles_instalados.
2) Estima qué porcentaje de la imagen ocupa la superficie de los paneles solares.
   Cuenta SOLO la superficie de las placas (los módulos oscuros).
   NO cuentes los pasillos de tierra entre filas, caminos, vallas, edificios
   ni el resto del recinto de la planta.
   Muchas imágenes tienen pocos paneles o ninguno; si no hay, responde 0.
Responde solo con un JSON así: {"fase": "...", "pct_paneles": 0-100}"""

PAUSA_SEG = 5
TAM_LOTE = 25     # guardo cada 25 teselas para no perder trabajo si se corta
TABLA_SALIDA = "Lh_Silver.silver_detecciones"

StatementMeta(, 9dba556d-42dd-40cc-b44d-db297855b545, 9, Finished, Available, Finished, False)

In [3]:
# Parto del inventario de Bronze y quito las teselas que ya están en Silver
teselas = spark.table("bronze_teselas")
if spark.catalog.tableExists(TABLA_SALIDA):
    hechas = spark.table(TABLA_SALIDA).select("anio", "tesela")
    teselas = teselas.join(hechas, ["anio", "tesela"], "left_anti")

pendientes = teselas.orderBy("anio", "fila", "columna").collect()
print(f"Teselas pendientes: {len(pendientes)}")

StatementMeta(, 9dba556d-42dd-40cc-b44d-db297855b545, 10, Finished, Available, Finished, False)

Teselas pendientes: 0


In [4]:
esquema = ("planta string, anio bigint, tesela string, fila bigint, columna bigint, "
           "fase string, pct_paneles double")

def preguntar_a_gemini(imagen):
    """Llama a Gemini. Si el servidor está saturado (503) espero y reintento."""
    for intento in range(3):
        try:
            r = client.models.generate_content(
                model=MODELO,
                contents=[imagen, PROMPT],
                config={"response_mime_type": "application/json"},
            )
            resp = json.loads(r.text)
            # A veces devuelve el JSON dentro de una lista: [{"fase": ..., ...}]
            if isinstance(resp, list):
                resp = resp[0]
            return resp
        except Exception as e:
            if "503" in str(e) and intento < 2:
                print("  Servidor saturado, espero 30 s...")
                time.sleep(30)
            else:
                raise

def guardar(lote):
    """Añade un lote de resultados a la tabla Silver."""
    (spark.createDataFrame(lote, schema=esquema)
         .withColumn("modelo", F.lit(MODELO))
         .withColumn("version_prompt", F.lit(VERSION_PROMPT))
         .withColumn("fecha_deteccion", F.current_timestamp())
         .write.format("delta").mode("append").saveAsTable(TABLA_SALIDA))

lote, guardadas = [], 0
for i, t in enumerate(pendientes, 1):
    imagen = Image.open(f"/lakehouse/default/{t.ruta}").convert("RGB")
    try:
        resp = preguntar_a_gemini(imagen)
        fila = {
            "planta": t.planta, "anio": t.anio, "tesela": t.tesela,
            "fila": t.fila, "columna": t.columna,
            "fase": resp["fase"], "pct_paneles": float(resp["pct_paneles"]),
        }
    except Exception as e:
        if "429" in str(e):
            print("Cuota diaria agotada. Lo que falta se procesará en la siguiente ejecución.")
            break
        # Respuesta rara o error puntual: me la salto, se reintentará en la próxima ejecución
        print(f"  Error en {t.anio}/{t.tesela}: {str(e)[:100]}")
        continue

    lote.append(fila)
    if len(lote) == TAM_LOTE:
        guardar(lote)
        guardadas += len(lote)
        lote = []
        print(f"{i}/{len(pendientes)} procesadas, {guardadas} guardadas")
    time.sleep(PAUSA_SEG)

# Guardo lo que quede en el último lote (también si he parado por la cuota)
if lote:
    guardar(lote)
    guardadas += len(lote)
print(f"Total guardadas en esta ejecución: {guardadas}")

StatementMeta(, 9dba556d-42dd-40cc-b44d-db297855b545, 11, Finished, Available, Finished, False)

Total guardadas en esta ejecución: 0


In [5]:
display(spark.sql(f"""
    SELECT anio,
           COUNT(*)                     AS teselas,
           ROUND(AVG(pct_paneles), 1)   AS pct_paneles_medio,
           SUM(CASE WHEN fase = 'paneles_instalados' THEN 1 ELSE 0 END)      AS con_paneles,
           SUM(CASE WHEN fase = 'estructuras_sin_paneles' THEN 1 ELSE 0 END) AS estructuras,
           SUM(CASE WHEN fase = 'movimiento_de_tierras' THEN 1 ELSE 0 END)   AS mov_tierras,
           SUM(CASE WHEN fase = 'terreno_sin_tocar' THEN 1 ELSE 0 END)       AS sin_tocar
    FROM {TABLA_SALIDA}
    GROUP BY anio ORDER BY anio
"""))

StatementMeta(, 9dba556d-42dd-40cc-b44d-db297855b545, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9f3a9952-6168-436c-a0fb-0abe27a2936f)